In [ ]:
from langchain.messages import HumanMessage,AIMessage,SystemMessage
from langchain.agents import create_agent
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START,END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import add_messages
from tavily import TavilyClient
from langchain.tools import tool
from typing import TypedDict, Annotated
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

In [5]:
load_dotenv()

True

In [6]:
memory = MemorySaver()

In [7]:
config = {"configurable":{"thread_id":"1"}}

In [8]:
llm = init_chat_model(model ="google_genai:gemini-3.5-flash")

In [6]:
@tool
def tavily_search(query:str)->dict:
    """search the web for information related to the query"""
    return TavilyClient.search(query=query)   

In [7]:
@tool
def add(a:int,b:int) ->int:
    """perform mathematic operation on a and b and return the value """
    return a*b


In [8]:
tools = [tavily_search,add]
llm = llm.bind_tools(tools)

In [9]:
class State(TypedDict):
    messages : Annotated[list, add_messages]

In [10]:
#node functionality
def llm_with_tools(state : State):
    system = SystemMessage(content="You are a helpful assistant. Use tools when needed but avoid redundant searches. One search should be sufficient for most questions.")
    return {"messages":[llm.invoke([system]+state['messages'])]}

In [11]:
builder = StateGraph(State)

In [12]:
builder.add_node("llm_main",llm_with_tools)
builder.add_node("tools",ToolNode(tools))

In [13]:
builder.add_edge(START, "llm_main")
builder.add_conditional_edges(
    "llm_main",
    #if the latest message from assistant is a tool call -> tool_condition results to tools
    #If the latest message from assitant is not a tool call -> tool_conditions routes to END"""
    tools_condition
)
builder.add_edge("tools","llm_main")

In [14]:
graph = builder.compile(checkpointer=memory)

Stream mode 'values' emits the entire state after each step

In [15]:
graph.invoke({"messages":"hi my name is namir and i like playing games"}, config=config, stream_mode='values')

{'messages': [HumanMessage(content='hi my name is namir and i like playing games', additional_kwargs={}, response_metadata={}, id='13fa41be-955f-4cd9-8ff1-a9e339706cdb'),
  AIMessage(content=[{'type': 'text', 'text': "Hi Namir! It's great to meet you. \n\nWhat kind of games do you like playing? Are you into video games, board games, tabletop RPGs, or maybe sports? I'd love to hear about your favorites!", 'extras': {'signature': 'Eu8GCuwGAWkUfRPmxxUpSfvXPIDyzWk3fPC9EWboqLTwhqwGI0jY8X/pGAxtqZBLcSPKi8czDV/TDa+jTNZDtvDBvdxN+xV/d6lzxB7/9agAwcs44HA7ZYInl7hXupXAsDxTC8FY3BTR/TXGVjrSBE3DICCGOsrOWh1Zf4JWpvl2LKWmdDe6V4zmZif2YMzJR4Lgw527tLgDE68PUnjynOK4ARTsolI8h/9J1imMSHeen33E/7kQY6BL972XzPMPYpxR6nKsZfdq/mNMKMR1AlyJa4qkDkDLn/mx3Hij+CXqhx9GsiZpG4ixHQyhZX8puTCuMQ7Bh0Cb/yU0ZaBEJBU3TNf6LxBfa5yn9y29mRpZeeeaZyUdt0dqIQ02+mALAFRHojmHUHENWKBO5UdKiTEvnBBZ5SFol5BrRXJ2iS+x5r88ksDghe5xKOe/ilZUg3dQ7jKOQ3atikrAfeeux/zKQ+F1yBhTgInvz3hz9sv+e8OrwjZb22iJr1PS6m3OuZDP566EOTEWfdtbXufLQRxIrVjflyjbjK0DLS5yIjPknORiDDo7BH0

In [16]:
print(graph.invoke({"messages":"i like playing story games"}, config=config, stream_mode='values'))

{'messages': [HumanMessage(content='hi my name is namir and i like playing games', additional_kwargs={}, response_metadata={}, id='13fa41be-955f-4cd9-8ff1-a9e339706cdb'), AIMessage(content=[{'type': 'text', 'text': "Hi Namir! It's great to meet you. \n\nWhat kind of games do you like playing? Are you into video games, board games, tabletop RPGs, or maybe sports? I'd love to hear about your favorites!", 'extras': {'signature': 'Eu8GCuwGAWkUfRPmxxUpSfvXPIDyzWk3fPC9EWboqLTwhqwGI0jY8X/pGAxtqZBLcSPKi8czDV/TDa+jTNZDtvDBvdxN+xV/d6lzxB7/9agAwcs44HA7ZYInl7hXupXAsDxTC8FY3BTR/TXGVjrSBE3DICCGOsrOWh1Zf4JWpvl2LKWmdDe6V4zmZif2YMzJR4Lgw527tLgDE68PUnjynOK4ARTsolI8h/9J1imMSHeen33E/7kQY6BL972XzPMPYpxR6nKsZfdq/mNMKMR1AlyJa4qkDkDLn/mx3Hij+CXqhx9GsiZpG4ixHQyhZX8puTCuMQ7Bh0Cb/yU0ZaBEJBU3TNf6LxBfa5yn9y29mRpZeeeaZyUdt0dqIQ02+mALAFRHojmHUHENWKBO5UdKiTEvnBBZ5SFol5BrRXJ2iS+x5r88ksDghe5xKOe/ilZUg3dQ7jKOQ3atikrAfeeux/zKQ+F1yBhTgInvz3hz9sv+e8OrwjZb22iJr1PS6m3OuZDP566EOTEWfdtbXufLQRxIrVjflyjbjK0DLS5yIjPknORiDDo7BH0rK

Stream mode 'updates' emits only the new data returned by that specific node, labeled by node name.

In [17]:
graph.invoke({"messages":"i have recently played marvel spiderman 2"}, config=config, stream_mode='updates')

[{'llm_main': {'messages': [AIMessage(content=[{'type': 'text', 'text': "Oh, *Marvel's Spider-Man 2* is a fantastic choice! Insomniac did an incredible job building on the first two games. The story gets so intense, especially with the Symbiote suit, Venom, and Kraven the Hunter.\n\nHow did you like the dynamic between Peter and Miles in this one? And did you have a favorite moment or mission from the game? (No spoilers if you want to keep it safe, but I'd love to hear what you thought of the story!)", 'extras': {'signature': 'EvQJCvEJAWkUfRMhczNHw1woHOHat8JcdtNfQzndBLdkbwrCIZup78KatHXc/y3ok4EEvWaKA9drJkOWGLjH/DuCvPqJd0r9hiFPyQHqXliN1PQl0f8gi8MVrmSW2m6H7Kj3R960yqieUNiPT1Q9I2zlCKQaWFnZMJd22OlN7RAUYGGScMXBshO/UQaYHjHJp31Fj0qovWOcC/qzYlrzYvAgzh4bGU5UtGbkvpGvpCxdvqgsoDHJvsgmgE6NG+lla9Yf+HaZXUqoqDei0CZ7BNlB7zRPoq4wMtv7F5x4iR2tjx0nGOm+w3XYi8vcGuEuVExrsEMQSMdgzmtLOQ1eO2T6xnltOEGOdFdf1sWY9TZuqdal/nxbTOyvJxMHfVXOv/ktApZK6a3k5TEjmBOjYgJdu3q3JSpLs0exg5vqwsnhT+4VjAFCaRPb4o1bCa7Zcpqmug/zyBJQ+PFbBTE

In [18]:
graph.invoke({"messages":"what do you think about god of war?"}, config=config, stream_mode='updates')

ChatGoogleGenerativeAIError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 15.355663825s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT